> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 3 · Notebook 08 — SQL, Parquet, validation and property tests

**Sessions:** S19–S20 (SQL, Parquet, DuckDB & data validation) · S22 (Testing in depth) · [Lesson plan](../../docs/lessons/PART_03_PYTHON_ENGINEERING.md) · graded labs in [`labs/part03/`](../../labs/part03/)

**You will:**
1. Store bars as partitioned Parquet and query them with DuckDB SQL.
2. Use a window function for a moving average per symbol.
3. Validate bars before anything trusts them.
4. Write a property-based test that catches a float bug.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p3lib.py is in notebooks/part03/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p3lib as p

p.use_course_style()
import duckdb, tempfile
import pyarrow as pa, pyarrow.parquet as pq

In [ ]:
rng = np.random.default_rng(0)
dates = pd.bdate_range("2024-01-02", periods=250)
frames = []
for i, sym in enumerate(["SPY", "QQQ", "IWM", "TLT", "GLD"]):
    close = 100 * (1 + i / 10) * np.exp(np.cumsum(rng.normal(0, 0.01, len(dates))))
    open_ = close * np.exp(rng.normal(0, 0.003, len(dates)))
    frames.append(pd.DataFrame({"date": dates, "symbol": sym, "open": open_, "close": close,
                                "high": np.maximum(open_, close) * 1.004, "low": np.minimum(open_, close) * 0.996,
                                "volume": rng.integers(1_000_000, 5_000_000, len(dates))}))
bars = pd.concat(frames, ignore_index=True)
root = Path(tempfile.mkdtemp()) / "bars"
pq.write_to_dataset(pa.Table.from_pandas(bars), root, partition_cols=["symbol"])      # hive: symbol=SPY/...
print(sorted(x.name for x in root.iterdir()))
con = duckdb.connect()
con.sql(f"CREATE VIEW bars AS SELECT * FROM read_parquet('{root}/**/*.parquet', hive_partitioning = true)")
con.sql("SELECT symbol, COUNT(*) AS n FROM bars GROUP BY symbol ORDER BY symbol").df()

## 1. SQL over Parquet

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
sql = """SELECT symbol, AVG(volume) AS avg_volume, MAX(close) AS max_close
FROM bars GROUP BY symbol ORDER BY symbol"""
got = con.sql(sql).df() if isinstance(sql, str) else None
ref = bars.groupby("symbol").agg(avg_volume=("volume", "mean"), max_close=("close", "max")).reset_index()
summary = p.check("SQL aggregation", got, ref)
summary

## 2. Window functions: a moving average per symbol

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
window = "PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW"
got = None
if isinstance(window, str):
    got = con.sql(f"SELECT symbol, date, AVG(close) OVER ({window}) AS sma20 FROM bars ORDER BY symbol, date").df()
    got = got.groupby("symbol").apply(lambda d: d.iloc[19:], include_groups=False)["sma20"].reset_index(drop=True)
ref = (bars.sort_values(["symbol", "date"]).groupby("symbol")["close"].rolling(20).mean().dropna()
       .reset_index(drop=True).rename("sma20"))
sma = p.check("SQL moving average", got, ref)

## 3. Validate before you trust

In [ ]:
dirty = bars.copy()
dirty.loc[10, "high"] = dirty.loc[10, "close"] * 0.9          # high below the close
dirty.loc[300, "low"] = dirty.loc[300, "open"] * 1.1          # low above the open
dirty.loc[777, "close"] = -1.0                                # impossible price
dirty.loc[900, "volume"] = -5                                  # impossible volume

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
bad = ((dirty["high"] < dirty[["open", "close"]].max(axis=1)) | (dirty["low"] > dirty[["open", "close"]].min(axis=1))
       | (dirty[["open", "high", "low", "close"]] <= 0).any(axis=1) | (dirty["volume"] < 0))
bad = p.check("bad-bar detector", bad, p.bad_bars(dirty))
dirty[bad]

## 4. Property-based testing with Hypothesis

Instead of a few hand-picked examples, state a **property** that must hold for every input and let Hypothesis search for a counterexample.

In [ ]:
from hypothesis import given, settings, strategies as st

def round_to_tick_float(price: Decimal, tick: Decimal) -> Decimal:
    """The buggy version: rounds in float, then turns the float back into a Decimal."""
    return Decimal(round(float(price) / float(tick)) * float(tick))

def passes(test) -> bool:
    try:
        test()
        return True
    except AssertionError:
        return False

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def make_test(round_fn):
    @settings(max_examples=300, deadline=None)
    @given(st.decimals(min_value=Decimal("0.01"), max_value=Decimal("10000"), places=4),
           st.sampled_from([Decimal("0.01"), Decimal("0.05"), Decimal("0.25")]))
    def test(price, tick):
        r = round_fn(price, tick)
        assert r % tick == 0
        assert abs(r - price) <= tick / 2
    return test

result = (passes(make_test(p.round_to_tick)), not passes(make_test(round_to_tick_float)))
result = p.check("property: decimal passes, float bug caught", result, (True, True))

## Questions
1. Why partition Parquet by symbol (and often by date)? What does DuckDB skip when you filter on `symbol`?
2. The first 19 rows of the SQL moving average are averages of fewer than 20 closes. How would you make them NULL instead?
3. Which other properties would you test for a `Position` class (hint: realized + unrealized P&L)?

**Graded version:** `labs/part03/week11_data` (bar validation, hive Parquet, DuckDB) and the property test in `week09_domain`.